# TT-13 — LASSO REGRESSION (L1)
## Chọn ra 10 chỉ số xét nghiệm quan trọng nhất trong 200 chỉ số

**Bài toán:** Bệnh viện muốn xây bộ xét nghiệm sàng lọc tiến triển bệnh tiểu đường
từ 200 chỉ số sinh hoá, nhưng chỉ muốn giữ lại ~10 chỉ số đủ tốt để dự đoán,
nhằm giảm chi phí xét nghiệm cho bệnh nhân.

Notebook này đi theo đúng 10 bước trong mục **5. CÁC BƯỚC THỰC HIỆN** của README.
Mỗi bước là 1 (hoặc vài) cell riêng biệt — chạy tuần tự từ trên xuống.


## 0. Import thư viện

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, LassoCV, Ridge, RidgeCV, lasso_path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.grid'] = True


## Bước 1 — Nạp `load_diabetes`, mở rộng thành 200 cột (10 thật + 190 nhiễu)

Ý tưởng: 10 cột gốc của `load_diabetes` là các chỉ số THẬT có tín hiệu.
Ta thêm 190 cột nhiễu ngẫu nhiên (Gaussian) để mô phỏng "200 chỉ số xét nghiệm",
trong đó chỉ 10 cột đầu là có ý nghĩa thật sự. Nhờ vậy ta **biết trước đáp án đúng**
để chấm điểm khả năng chọn biến của Lasso.

In [ ]:
X, y = load_diabetes(return_X_y=True, as_frame=True)

BIEN_THAT = list(X.columns)          # 10 tên cột thật (age, sex, bmi, bp, s1..s6)
print("10 biến THẬT:", BIEN_THAT)

# Thêm 190 cột NHIỄU thuần tuý
rng = np.random.default_rng(RANDOM_STATE)
nhieu = pd.DataFrame(
    rng.normal(size=(len(X), 190)),
    columns=[f'chi_so_nhieu_{i:03d}' for i in range(190)]
)

X_full = pd.concat([X, nhieu], axis=1)      # 200 cột, chỉ 10 cột có tín hiệu thật
BIEN_NHIEU = list(nhieu.columns)

print(f"Kích thước X_full: {X_full.shape}  (kỳ vọng: (442, 200))")
X_full.head()


In [ ]:
# Chia train/test — dùng chung cho toàn bộ notebook
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")


## Bước 2 — Baseline: Linear Regression trên 200 cột → quan sát overfit nặng

Với p=200 biến nhưng chỉ ~354 mẫu train (và chỉ 10 biến có tín hiệu thật),
Linear Regression không hề co hệ số nào về 0 → dễ overfit: RMSE train thấp
nhưng RMSE test cao hơn hẳn.

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

rmse_train_lr = np.sqrt(mean_squared_error(y_train, lr.predict(X_train)))
rmse_test_lr  = np.sqrt(mean_squared_error(y_test, lr.predict(X_test)))

print(f"Linear Regression (baseline, 200 biến):")
print(f"  RMSE train = {rmse_train_lr:.3f}")
print(f"  RMSE test  = {rmse_test_lr:.3f}")
print(f"  Số hệ số != 0: {(lr.coef_ != 0).sum()}/200  (Linear Regression không loại biến nào)")
print(f"  → Chênh lệch train/test càng lớn thì overfit càng nặng.")


## Bước 3 — LassoCV dò alpha

**Bắt buộc chuẩn hoá (StandardScaler)** trước khi đưa vào Lasso, vì phạt L1 phụ
thuộc vào độ lớn hệ số — biến có đơn vị lớn/nhỏ khác nhau sẽ bị phạt không công bằng
nếu không scale.

In [ ]:
pipe = Pipeline([
    ('scale', StandardScaler()),               # BẮT BUỘC
    ('lasso', LassoCV(alphas=np.logspace(-4, 1, 100), cv=5,
                      max_iter=50000, random_state=RANDOM_STATE)),
])
pipe.fit(X_train, y_train)

alpha_toi_uu = pipe['lasso'].alpha_
he_so = pipe['lasso'].coef_
so_bien_giu = (he_so != 0).sum()

print(f"Alpha tối ưu (theo CV): {alpha_toi_uu:.5f}")
print(f"Lasso giữ lại {so_bien_giu}/200 biến")

rmse_train_lasso = np.sqrt(mean_squared_error(y_train, pipe.predict(X_train)))
rmse_test_lasso  = np.sqrt(mean_squared_error(y_test, pipe.predict(X_test)))
print(f"RMSE train = {rmse_train_lasso:.3f} | RMSE test = {rmse_test_lasso:.3f}")


## Bước 4 — ⭐ Chấm điểm chọn biến

- **Recall chọn biến**: trong 10 biến THẬT, Lasso giữ lại bao nhiêu?
- **False positive**: trong số biến nhiễu, bao nhiêu bị giữ nhầm?

In [ ]:
bien_duoc_giu = X_full.columns[he_so != 0].tolist()

that_duoc_giu   = [b for b in bien_duoc_giu if b in BIEN_THAT]
nhieu_bi_giu_nham = [b for b in bien_duoc_giu if b in BIEN_NHIEU]

bang_cham_diem = pd.DataFrame({
    'Chỉ số': [
        'Số biến THẬT được giữ (recall)',
        'Số biến NHIỄU bị giữ nhầm (false positive)',
        'Tổng số biến Lasso giữ lại',
        'Alpha tối ưu',
    ],
    'Giá trị': [
        f"{len(that_duoc_giu)}/10",
        f"{len(nhieu_bi_giu_nham)}/190",
        f"{so_bien_giu}/200",
        f"{alpha_toi_uu:.5f}",
    ]
})

print("BẢNG CHẤM ĐIỂM CHỌN BIẾN")
display(bang_cham_diem)

print("\nCác biến THẬT được giữ:   ", that_duoc_giu)
print("Các biến THẬT bị loại:    ", [b for b in BIEN_THAT if b not in that_duoc_giu])
print("Vài biến NHIỄU bị giữ nhầm (nếu có):", nhieu_bi_giu_nham[:10])

# Đối chiếu trung thực với mức tham chiếu trong README (6-9/10 biến thật)
# — KHÔNG che giấu nếu kết quả thực tế thấp hơn.
if len(that_duoc_giu) < 6:
    print(f"\n⚠️ Chỉ bắt lại được {len(that_duoc_giu)}/10 biến thật — THẤP hơn mức tham chiếu "
          f"(6-9/10) nêu trong README. Đây là kết quả thực tế của lần chạy này, không nên "
          f"chỉnh alpha/seed để 'đẹp' số liệu hơn. Nguyên nhân khả dĩ: bộ diabetes vốn có "
          f"tín hiệu yếu, và alpha do CV chọn ({alpha_toi_uu:.2f}) khá lớn nên Lasso phạt mạnh, "
          f"loại luôn vài biến thật có tín hiệu yếu để đổi lấy mô hình thưa hơn.")
if len(nhieu_bi_giu_nham) > 0:
    print(f"\n⚠️ Có {len(nhieu_bi_giu_nham)} biến NHIỄU bị giữ nhầm — cần đánh dấu rõ trong "
          f"danh sách đề xuất cuối cùng (Bước 10) để không đưa nhầm vào bộ xét nghiệm thực tế.")


## Bước 5 — Vẽ coefficient path của Lasso

Coefficient path cho thấy khi alpha (độ phạt) tăng dần, các hệ số lần lượt
"rơi" về đúng 0 — trực quan hoá đúng cơ chế chọn biến của Lasso.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

alphas_path, coefs_path, _ = lasso_path(
    X_train_scaled, y_train,
    alphas=np.logspace(-4, 1, 100),
    max_iter=50000,
)

fig, ax = plt.subplots(figsize=(10, 6))
for i, ten_bien in enumerate(X_full.columns):
    mau = 'crimson' if ten_bien in BIEN_THAT else 'lightgray'
    do_day = 2 if ten_bien in BIEN_THAT else 0.6
    zorder = 3 if ten_bien in BIEN_THAT else 1
    ax.plot(np.log10(alphas_path), coefs_path[i], color=mau, linewidth=do_day, zorder=zorder)

ax.axvline(np.log10(alpha_toi_uu), color='blue', linestyle='--', label=f'alpha tối ưu (CV) = {alpha_toi_uu:.4f}')
ax.set_xlabel('log10(alpha)')
ax.set_ylabel('Hệ số (đã chuẩn hoá)')
ax.set_title('Coefficient path của Lasso\n(đỏ = 10 biến thật, xám = 190 biến nhiễu)')
ax.legend()
plt.tight_layout()
plt.savefig('lasso_path.png', dpi=120)
plt.show()


## Bước 6 — Vẽ RMSE train/test theo alpha

In [ ]:
alphas_ve = np.logspace(-4, 1, 60)
rmse_train_list, rmse_test_list = [], []

for a in alphas_ve:
    mo_hinh = Pipeline([
        ('scale', StandardScaler()),
        ('lasso', Lasso(alpha=a, max_iter=50000, random_state=RANDOM_STATE)),
    ])
    mo_hinh.fit(X_train, y_train)
    rmse_train_list.append(np.sqrt(mean_squared_error(y_train, mo_hinh.predict(X_train))))
    rmse_test_list.append(np.sqrt(mean_squared_error(y_test, mo_hinh.predict(X_test))))

fig, ax = plt.subplots()
ax.plot(np.log10(alphas_ve), rmse_train_list, label='RMSE train', marker='o', markersize=3)
ax.plot(np.log10(alphas_ve), rmse_test_list, label='RMSE test', marker='o', markersize=3)
ax.axvline(np.log10(alpha_toi_uu), color='blue', linestyle='--', label='alpha tối ưu (CV)')
ax.set_xlabel('log10(alpha)')
ax.set_ylabel('RMSE')
ax.set_title('RMSE train/test theo alpha (Lasso)')
ax.legend()
plt.tight_layout()
plt.show()


## Bước 7 — So sánh Ridge vs Lasso trên cùng dữ liệu

Kỳ vọng: Ridge giữ lại **cả 200** biến (chỉ co nhỏ hệ số, không đưa về đúng 0),
trong khi Lasso loại bỏ phần lớn biến nhiễu.

In [ ]:
pipe_ridge = Pipeline([
    ('scale', StandardScaler()),
    ('ridge', RidgeCV(alphas=np.logspace(-4, 4, 100), cv=5)),
])
pipe_ridge.fit(X_train, y_train)

he_so_ridge = pipe_ridge['ridge'].coef_
so_bien_giu_ridge = (he_so_ridge != 0).sum()   # gần như luôn = 200

rmse_train_ridge = np.sqrt(mean_squared_error(y_train, pipe_ridge.predict(X_train)))
rmse_test_ridge  = np.sqrt(mean_squared_error(y_test, pipe_ridge.predict(X_test)))

bang_so_sanh = pd.DataFrame({
    'Mô hình': ['Linear Regression (baseline)', 'Ridge (L2)', 'Lasso (L1)'],
    'Số biến giữ lại': [(lr.coef_ != 0).sum(), so_bien_giu_ridge, so_bien_giu],
    'RMSE train': [rmse_train_lr, rmse_train_ridge, rmse_train_lasso],
    'RMSE test':  [rmse_test_lr,  rmse_test_ridge,  rmse_test_lasso],
})

print("BẢNG SO SÁNH RIDGE vs LASSO vs LINEAR REGRESSION")
display(bang_so_sanh)


## Bước 8 — ⚠️ Thí nghiệm biến tương quan

Nhân đôi 1 cột thật (ví dụ `bmi`) và thêm nhiễu nhỏ để tạo ra 2 biến tương quan
rất cao (~0.95+). Lasso sẽ chỉ giữ **1 trong 2 biến gần như ngẫu nhiên** —
minh hoạ đúng điểm yếu nêu trong README.

In [ ]:
bien_can_nhan_ban = 'bmi'

def tao_du_lieu_trung_lap(X_goc, seed):
    rng_local = np.random.default_rng(seed)
    X_moi = X_goc.copy()
    nhieu_nho = rng_local.normal(scale=0.01 * X_goc[bien_can_nhan_ban].std(), size=len(X_goc))
    X_moi[f'{bien_can_nhan_ban}_ban_sao'] = X_goc[bien_can_nhan_ban] + nhieu_nho
    return X_moi

ket_qua_thi_nghiem = []
for seed in [1, 2, 3, 42, 100]:
    X_full_dup = tao_du_lieu_trung_lap(X_full, seed)
    X_tr, X_te, y_tr, y_te = train_test_split(X_full_dup, y, test_size=0.2, random_state=RANDOM_STATE)

    # Lưới alpha nhẹ hơn Bước 3 (chỉ để minh hoạ tính KHÔNG ổn định, không cần
    # dò alpha thật mịn) — giúp thí nghiệm 5 lần chạy nhanh hơn nhiều.
    pipe_dup = Pipeline([
        ('scale', StandardScaler()),
        ('lasso', LassoCV(alphas=np.logspace(-3, 1, 30), cv=3,
                          max_iter=20000, random_state=RANDOM_STATE)),
    ])
    pipe_dup.fit(X_tr, y_tr)

    he_so_dup = pd.Series(pipe_dup['lasso'].coef_, index=X_full_dup.columns)
    goc = he_so_dup[bien_can_nhan_ban]
    ban_sao = he_so_dup[f'{bien_can_nhan_ban}_ban_sao']

    ket_qua_thi_nghiem.append({
        'seed_tao_nhieu': seed,
        f'he_so_{bien_can_nhan_ban}_goc': round(goc, 4),
        f'he_so_{bien_can_nhan_ban}_ban_sao': round(ban_sao, 4),
        'bien_duoc_chon': bien_can_nhan_ban if abs(goc) > abs(ban_sao) else f'{bien_can_nhan_ban}_ban_sao'
                          if (goc != 0 or ban_sao != 0) else 'không biến nào'
    })

bang_thi_nghiem = pd.DataFrame(ket_qua_thi_nghiem)
print("KẾT QUẢ THÍ NGHIỆM BIẾN TƯƠNG QUAN (chạy lại với 5 seed khác nhau)")
display(bang_thi_nghiem)
print("\n→ Nhận xét: nếu cột 'bien_duoc_chon' đổi qua các seed, chứng tỏ lựa chọn")
print("  của Lasso KHÔNG ổn định khi có biến tương quan cao — đúng như README cảnh báo.")


## Bước 9 — Train lại Linear Regression CHỈ trên các biến Lasso chọn (debiased lasso)

Lasso co hệ số về gần 0 nên có thể bị **thiên lệch (biased)**. Kỹ thuật "debiased lasso"
(hay "relaxed lasso"): lấy đúng tập biến mà Lasso đã chọn, rồi fit lại bằng
Linear Regression thường (không phạt) trên đúng tập biến đó, xem RMSE có cải thiện không.

In [ ]:
bien_lasso_chon = X_full.columns[he_so != 0].tolist()
print(f"Số biến được Lasso chọn: {len(bien_lasso_chon)}")

lr_debiased = LinearRegression()
lr_debiased.fit(X_train[bien_lasso_chon], y_train)

rmse_train_debiased = np.sqrt(mean_squared_error(y_train, lr_debiased.predict(X_train[bien_lasso_chon])))
rmse_test_debiased  = np.sqrt(mean_squared_error(y_test, lr_debiased.predict(X_test[bien_lasso_chon])))

bang_debiased = pd.DataFrame({
    'Mô hình': ['Lasso đầy đủ (200 biến, hệ số bị co)', 'Debiased Lasso (chỉ biến được chọn, không phạt)'],
    'Số biến': [200, len(bien_lasso_chon)],
    'RMSE train': [rmse_train_lasso, rmse_train_debiased],
    'RMSE test':  [rmse_test_lasso,  rmse_test_debiased],
})
print("SO SÁNH LASSO ĐẦY ĐỦ vs DEBIASED LASSO")
display(bang_debiased)


## Mở rộng — Stability Selection

Bước 8 cho thấy Lasso không ổn định khi có biến tương quan cao. README đề xuất
khắc phục bằng **Stability Selection**: chạy Lasso nhiều lần trên các mẫu
bootstrap của tập train, chỉ giữ lại biến nào được chọn (hệ số ≠ 0) trong
**hơn 60% số lần chạy** — biến "may mắn" được chọn 1 lần do tương quan ngẫu
nhiên sẽ bị lọc bỏ vì không lặp lại ổn định qua các mẫu bootstrap khác nhau.

In [ ]:
from sklearn.utils import resample

SO_LAN_BOOTSTRAP = 100
NGUONG_ON_DINH = 0.6

dem_duoc_chon = pd.Series(0, index=X_full.columns)

for lan in range(SO_LAN_BOOTSTRAP):
    X_boot, y_boot = resample(X_train, y_train, random_state=lan)
    pipe_boot = Pipeline([
        ('scale', StandardScaler()),
        ('lasso', Lasso(alpha=alpha_toi_uu, max_iter=20000, random_state=lan)),
    ])
    pipe_boot.fit(X_boot, y_boot)
    dem_duoc_chon += (pipe_boot['lasso'].coef_ != 0).astype(int)

ty_le_on_dinh = (dem_duoc_chon / SO_LAN_BOOTSTRAP).sort_values(ascending=False)
bien_on_dinh = ty_le_on_dinh[ty_le_on_dinh > NGUONG_ON_DINH]

print(f"Stability Selection: {len(bien_on_dinh)} biến được chọn > {NGUONG_ON_DINH*100:.0f}% "
      f"số lần bootstrap (trong {SO_LAN_BOOTSTRAP} lần)")
display(bien_on_dinh.to_frame('Tỷ lệ được chọn'))

that_on_dinh = [b for b in bien_on_dinh.index if b in BIEN_THAT]
nhieu_on_dinh = [b for b in bien_on_dinh.index if b in BIEN_NHIEU]
print(f"\nTrong đó có {len(that_on_dinh)}/10 là biến THẬT: {that_on_dinh}")
print(f"Có {len(nhieu_on_dinh)} biến NHIỄU vẫn lọt qua ngưỡng ổn định: {nhieu_on_dinh}")
print(f"\n→ So với Bước 4 (Lasso chạy 1 lần): so sánh xem Stability Selection có giúp "
      f"loại bớt biến nhiễu 'may mắn' hay không, và có bỏ sót thêm biến thật nào không.")


## Bước 10 — ✍️ Đề xuất bộ xét nghiệm cuối cùng + ước tính chi phí tiết kiệm

Giả định mỗi chỉ số xét nghiệm tốn trung bình trong khoảng 30.000–200.000đ
(lấy theo README). Tính chi phí ước tính khi đo đủ 200 chỉ số so với chỉ đo
các chỉ số mà Lasso chọn.

In [ ]:
GIA_MOI_CHI_SO_MIN = 30_000
GIA_MOI_CHI_SO_MAX = 200_000
GIA_TRUNG_BINH = (GIA_MOI_CHI_SO_MIN + GIA_MOI_CHI_SO_MAX) / 2

so_chi_so_de_xuat = len(bien_lasso_chon)

chi_phi_200 = 200 * GIA_TRUNG_BINH
chi_phi_de_xuat = so_chi_so_de_xuat * GIA_TRUNG_BINH
tiet_kiem = chi_phi_200 - chi_phi_de_xuat
ty_le_tiet_kiem = tiet_kiem / chi_phi_200 * 100

print("ĐỀ XUẤT BỘ XÉT NGHIỆM CUỐI CÙNG")
print("=" * 50)
print(f"Danh sách {so_chi_so_de_xuat} chỉ số được đề xuất:")
for b in bien_lasso_chon:
    loai = "THẬT (tín hiệu y khoa)" if b in BIEN_THAT else "NHIỄU (nên xem lại/loại bỏ)"
    print(f"  - {b:25s} [{loai}]")

print("\nƯỚC TÍNH CHI PHÍ (đơn giá trung bình mỗi chỉ số ~ {:,.0f}đ)".format(GIA_TRUNG_BINH))
print(f"  Chi phí nếu đo đủ 200 chỉ số : {chi_phi_200:,.0f}đ / bệnh nhân")
print(f"  Chi phí với bộ đề xuất       : {chi_phi_de_xuat:,.0f}đ / bệnh nhân")
print(f"  → Tiết kiệm: {tiet_kiem:,.0f}đ / bệnh nhân ({ty_le_tiet_kiem:.1f}%)")


## Tổng kết — Vì sao hình thoi (Lasso) đưa hệ số về đúng 0 còn hình tròn (Ridge) thì không?

- Ràng buộc L1 (`Σ|wᵢ| ≤ t`) tạo ra một vùng khả thi hình **thoi** (kim cương),
  có các **góc nhọn nằm chính xác trên các trục toạ độ** (nơi một hay nhiều
  hệ số = 0).
- Ràng buộc L2 (`Σwᵢ² ≤ t`) tạo ra vùng khả thi hình **tròn/elip**, không có góc —
  đường bao trơn mọi nơi.
- Khi tìm điểm tối ưu (nơi đường đồng mức của hàm mất mát chạm vào vùng ràng buộc),
  với hình thoi, điểm chạm **rất hay rơi đúng vào góc** (vì góc là điểm "lồi cực đoan"
  dễ bị chạm nhất) → góc nằm trên trục → hệ số tương ứng = 0 chính xác.
- Với hình tròn, gần như không có điểm nào trên biên "đặc biệt" hơn điểm nào khác,
  nên điểm chạm gần như luôn nằm ở toạ độ khác 0 trên mọi trục → chỉ co nhỏ, không
  bao giờ chạm đúng 0.

### Bảng chấm điểm chọn biến (tóm tắt)


In [ ]:
print("== TÓM TẮT KẾT QUẢ ==")
display(bang_cham_diem)
display(bang_so_sanh)

print(f"\nLasso giữ {so_bien_giu}/200 biến, bắt lại {len(that_duoc_giu)}/10 biến thật, "
      f"giữ nhầm {len(nhieu_bi_giu_nham)}/190 biến nhiễu.")
if len(that_duoc_giu) < 6:
    print(f"→ {len(that_duoc_giu)}/10 THẤP hơn mức tham chiếu 6-9/10 trong README — kết quả "
          f"này được giữ nguyên trung thực, không chỉnh sửa cho 'đẹp'.")
print(f"\nVề RMSE: Lasso ({rmse_test_lasso:.2f}) tổng quát tốt hơn hẳn Linear Regression đầy đủ "
      f"200 biến ({rmse_test_lr:.2f}) dù Lasso chỉ dùng {so_bien_giu} biến — minh chứng rõ cho "
      f"nguyên lý 'mô hình đơn giản hơn tổng quát tốt hơn' khi p gần bằng hoặc lớn hơn n.")


### Nhận xét chốt

- **Trung thực về kết quả chọn biến**: kết quả thực tế của Lasso (xem số liệu ở
  bảng chấm điểm và dòng in phía trên) được giữ nguyên, kể cả khi thấp hơn mức
  tham chiếu 6–9/10 nêu trong README. Số biến nhiễu bị giữ nhầm (nếu có) cũng
  được liệt kê rõ và đánh dấu "nên xem lại" trong danh sách đề xuất ở Bước 10,
  thay vì âm thầm loại bỏ khỏi báo cáo.
- **Đánh đổi giữa độ thưa và độ nhạy**: alpha do `LassoCV` chọn tối ưu hoá RMSE
  trên cross-validation, không tối ưu hoá "bắt đúng 10 biến thật" — vì vậy mô
  hình có thể rất thưa (RMSE tốt) nhưng bỏ sót vài biến thật có tín hiệu yếu.
  Đây là lý do Stability Selection ở trên hữu ích: nó cho một góc nhìn khác về
  độ tin cậy của từng biến, độc lập với việc chọn đúng 1 giá trị alpha.
- **Tính không ổn định với biến tương quan** (Bước 8) là hạn chế cố hữu của
  Lasso, không phải lỗi cài đặt — đây chính là động lực ra đời của ElasticNet
  (TT-14).


## Xuất file sản phẩm nộp bài

Cell dưới đây tạo các file theo đúng cấu trúc mục 8 README, giả định các thư
mục `models/`, `reports/`, `src/` đã có sẵn cạnh notebook này (không tự tạo
thư mục). Mỗi hình được **vẽ lại từ đầu trong chính cell này** rồi mới lưu, để
tránh lỗi lưu ra ảnh trống do gọi `savefig` sau khi hình đã `show()`/đóng ở
cell trước.

In [ ]:
import glob, os, subprocess, sys
import joblib

# Dọn file rác lỡ ghi nhầm vào thư mục gốc ở các lần chạy trước (nếu có)
for ten_file_rac in ["lasso_pipeline.joblib", "lasso_path.png", "ridge_vs_lasso.png",
                     "chon_bien_score.png", "train.py", "requirements.txt"]:
    if os.path.exists(ten_file_rac):
        os.remove(ten_file_rac)
        print(f"Đã xoá file rác ở thư mục gốc: {ten_file_rac}")

# 1) Model
joblib.dump(pipe, "models/lasso_pipeline.joblib")

# 2) reports/lasso_path.png — vẽ lại đầy đủ, không phụ thuộc trạng thái figure cũ
fig, ax = plt.subplots(figsize=(10, 6))
for i, ten_bien in enumerate(X_full.columns):
    mau = 'crimson' if ten_bien in BIEN_THAT else 'lightgray'
    do_day = 2 if ten_bien in BIEN_THAT else 0.6
    zorder = 3 if ten_bien in BIEN_THAT else 1
    ax.plot(np.log10(alphas_path), coefs_path[i], color=mau, linewidth=do_day, zorder=zorder)
ax.axvline(np.log10(alpha_toi_uu), color='blue', linestyle='--',
           label=f'alpha tối ưu (CV) = {alpha_toi_uu:.4f}')
ax.set_xlabel('log10(alpha)')
ax.set_ylabel('Hệ số (đã chuẩn hoá)')
ax.set_title('Coefficient path của Lasso\n(đỏ = 10 biến thật, xám = 190 biến nhiễu)')
ax.legend()
fig.tight_layout()
fig.savefig("reports/lasso_path.png", dpi=120)
plt.close(fig)

# 3) reports/ridge_vs_lasso.png
fig, ax = plt.subplots()
bang_so_sanh.plot(x='Mô hình', y=['RMSE train', 'RMSE test'], kind='bar', ax=ax)
ax.set_title('So sánh RMSE: Linear vs Ridge vs Lasso')
fig.tight_layout()
fig.savefig("reports/ridge_vs_lasso.png", dpi=120)
plt.close(fig)

# 4) reports/chon_bien_score.png
fig, ax = plt.subplots(figsize=(6, 2))
ax.axis('off')
ax.table(cellText=bang_cham_diem.values, colLabels=bang_cham_diem.columns,
          loc='center', cellLoc='left')
fig.tight_layout()
fig.savefig("reports/chon_bien_score.png", dpi=120, bbox_inches='tight')
plt.close(fig)

print("Đã lưu: models/lasso_pipeline.joblib, reports/lasso_path.png, "
      "reports/ridge_vs_lasso.png, reports/chon_bien_score.png")

# 5) src/train.py — TỰ nhận diện tên file notebook hiện tại, tránh lỗi trỏ
#    sai tên khiến nbconvert âm thầm không sinh ra file (lỗi cũ của bản trước)
cac_file_ipynb = glob.glob("*.ipynb")
if len(cac_file_ipynb) == 1:
    ten_notebook = cac_file_ipynb[0]
    ket_qua = subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "script",
                               ten_notebook, "--output-dir", "src", "--output", "train"])
    # Một số phiên bản nbconvert xuất ra train.txt thay vì train.py — đổi tên lại cho đúng.
    duong_dan_txt = os.path.join("src", "train.txt")
    duong_dan_py = os.path.join("src", "train.py")
    if os.path.exists(duong_dan_txt):
        os.replace(duong_dan_txt, duong_dan_py)
    if ket_qua.returncode == 0 and os.path.exists(duong_dan_py):
        print(f"Đã xuất src/train.py từ notebook: {ten_notebook}")
    else:
        print("⚠️ nbconvert chạy lỗi hoặc không tạo được src/train.py — kiểm tra lại cài đặt jupyter/nbconvert.")
else:
    print(f"⚠️ Tìm thấy {len(cac_file_ipynb)} file .ipynb trong thư mục hiện tại — "
          f"không tự chọn được file nào. Hãy tự chạy lệnh sau, thay đúng tên notebook:")
    print("   jupyter nbconvert --to script <ten_notebook_cua_ban>.ipynb --output-dir src --output train")

# 6) requirements.txt — CHỈ liệt kê đúng thư viện thực sự dùng trong notebook,
#    KHÔNG dùng `pip freeze` (tránh kéo theo hàng chục gói không liên quan
#    như asttokens/colorama của môi trường Jupyter).
thu_vien_da_dung = ["numpy", "pandas", "matplotlib", "scikit-learn", "joblib"]
with open("requirements.txt", "w") as f:
    f.write("\n".join(thu_vien_da_dung) + "\n")

print("Đã lưu: requirements.txt (chỉ gồm thư viện thực sự dùng)")
